In [ ]:
import * as tslab from "tslab";
import { readFileSync } from "fs";

const css = readFileSync("../style.css", "utf-8");
tslab.display.html(`<style>${css}</style>`);

# Radix Sort

As <em style="color:blue">radix sort</em> is based on <em style="color:blue">counting sort</em>, we have to start our implementation of *radix sort* by defining the function `countingSort` that we have already discussed previously.

The function $\texttt{extractByte}(n, k)$ takes a natural number $n < 2^{32}$ and a number $k\in \{1,2,3,4\}$ as arguments.  It returns the $k$-th byte of $n$. 

In [ ]:
function extractByte(n: number, k: number): number {
  const u = n >>> 0;
  return (u >>> (8 * k)) & 0xff;
}

In [ ]:
const n = 123456789;
const B = Array.from({ length: 4 }, (_, k) => extractByte(n, k));
console.log(B); 

const reconstructed = B.reduce((acc, b, k) => acc + (b << (8 * k)), 0) >>> 0;
console.assert(reconstructed === (n >>> 0), "Reconstruction failed");

The function $\texttt{radixSort}(L)$ sorts an array $L$ of unsigned 32 bit integers and returns the sorted array.
The idea is to sort these numbers by first sorting them with respect to their last byte, then to sort the array with respect to the second byte, then with respect to the third byte, and finally with respect to the most important byte.
These four sorts are done using <em style="color:blue">counting sort</em>.
The fact that <em style="color:blue">counting sort</em> is <em style="color:blue">stable</em> guarantees that when we sort with respect to the second byte, numbers that have the same second byte will still be sorted with respect to the first byte.

In [ ]:
import { countingSort } from './Counting-Sort';

In [ ]:
function radixSort(L: number[]): number[] {
  let A: [number, number][] = L.map(n => [n >>> 0, 0]);

  for (let k = 0; k <= 3; k++) {
    A = A.map(([n, _]) => [n, extractByte(n, k)]);

    A = countingSort(A);
  }

  return A.map(([n, _]) => n >>> 0);
}

## Testing

In [ ]:
function demo() {
  const L: number[] = Array.from({ length: 15 }, () => Math.floor(Math.random() * 999) + 1);
  console.log("L =", L);
  let S = radixSort(L);
  console.log("S =", S);
}

In [ ]:
demo();

In [ ]:
function isOrdered(L: number[]): void {
  for (let i = 0; i < L.length - 1; i++) {
    if (L[i] > L[i + 1]) {
      throw new Error(`${L} not ordered at ${i}`);
    }
  }
}

The function `counter` takes an array as input and returns a Map that keeps count of how many times each item occurs in the array.

In [ ]:
function counter<T>(arr: T[]): Map<T, number> {
  const counts = new Map<T, number>();
  for (const item of arr) {
    counts.set(item, (counts.get(item) ?? 0) + 1);
  }
  return counts;
}

We also define the helper function `compareCounter` to be able to compare the contents of two counters.

In [ ]:

function compareCounters(a: Map<number, number>, b: Map<number, number>): boolean {
  if (a.size !== b.size) return false;
  for (const [key, value] of a) {
    if (b.get(key) !== value) return false;
  }
  return true;
}


The function `sameElements(L, S)` returns `true` if the arrays `L` and `S` contain the same elements and, furthermore, each 
element $x$ occurring in `L` occurs in `S` the same number of times it occurs in `L`.

In [ ]:
import assert from 'assert';

function sameElements(L: number[], S: number[]): void {
  assert(compareCounters(counter(L), counter(S)), "L and S do not have the same elements");
}

The function `randomIntRange(min, max)` generates a random integer in the range `[min, max)` and corresponds to Python's `range(min, max)` behavior regarding the exclusive upper bound.

In [ ]:
function randomIntRange(min: number, max: number): number {
  return Math.floor(Math.random() * (max - min)) + min;
} 

The function $\texttt{testSort}(n, k)$ generates $n$ random lists of length $k$, sorts them, and checks whether the output is sorted and contains the same elements as the input.

In [ ]:
function testSort(n: number, k: number): void {
  for (let i = 0; i < n; i++) {
    let L = Array.from({ length: k }, () => randomIntRange(0, 2 * k));
    const oldL = [...L];
    L = radixSort(L);
    isOrdered(L);
    sameElements(oldL, L);
    console.assert(L.length === oldL.length, "Array length changed");
    process.stdout.write(".");
  }
  console.log("\nAll tests successful!");
}

In [ ]:
console.time("testSort");
testSort(100, 20000);
console.timeEnd("testSort");

In [ ]:
console.time("1 million random integers");
const k = 1_000_000;
const L = Array.from({ length: k }, () => randomIntRange(0, 2 ** 32 - 1));
radixSort(L);
console.timeEnd("1 million random integers");